# Phát hiện Đặc trưng và Đối sánh Ảnh - Phần 2

**Sinh viên thực hiện:** [Họ tên sinh viên]  
**Lớp:** [Tên lớp]  
**Môn học:** Thị giác máy tính / Computer Vision  

---

Trong bài thực hành này, chúng ta sẽ thực hiện các nội dung chính sau:
- Đối sánh ảnh bằng phương pháp truyền thống (ORB, SIFT, BFMatcher, FLANN)
- Cải thiện kết quả bằng ratio test và RANSAC
- Xây dựng CNN trích xuất đặc trưng bằng PyTorch
- Xây dựng Siamese Network
- Huấn luyện với Contrastive Loss
- Nhận diện khuôn mặt thời gian thực bằng MTCNN + FaceNet

---
## Phần 1: Chuẩn bị môi trường

In [1]:
# Kiểm tra import thư viện

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import torch
import torchvision
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

print(f'OpenCV   : {cv2.__version__}')
print(f'NumPy    : {np.__version__}')
print(f'Matplotlib: {matplotlib.__version__}')
print(f'PyTorch  : {torch.__version__}')
print(f'Torchvision: {torchvision.__version__}')
print(f'GPU available: {torch.cuda.is_available()}')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Sử dụng thiết bị: {device}')

ModuleNotFoundError: No module named 'torch.utils'

---
## Phần 2: Đối sánh ảnh bằng phương pháp truyền thống

Ở phần này, chúng ta sẽ thực hiện đối sánh đặc trưng (feature matching) giữa 2 ảnh sử dụng thuật toán **ORB** (Oriented FAST and Rotated BRIEF) kết hợp với **Brute-Force Matcher**.

**ORB** là thuật toán phát hiện đặc trưng miễn phí bản quyền, nhanh hơn SIFT/SURF, phù hợp sử dụng trong thực tế.

In [ ]:
import urllib.request
import os

# Tạo 2 ảnh demo đơn giản bằng numpy (để tránh phụ thuộc file ngoài)
# Ảnh 1: nền trắng, hình chữ nhật đen ở giữa
img1 = np.ones((400, 600, 3), dtype=np.uint8) * 200
cv2.rectangle(img1, (100, 80), (500, 320), (30, 30, 30), -1)
cv2.circle(img1, (300, 200), 60, (180, 50, 50), -1)
cv2.putText(img1, 'Image A', (220, 380), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0,0,0), 2)

# Ảnh 2: xoay nhẹ + thêm nhiễu để giả lập ảnh thứ 2
img2 = np.ones((400, 600, 3), dtype=np.uint8) * 210
M = cv2.getRotationMatrix2D((300, 200), 12, 0.9)
img2 = cv2.warpAffine(img1, M, (600, 400))
noise = np.random.randint(0, 30, img2.shape, dtype=np.uint8)
img2 = cv2.add(img2, noise)
cv2.putText(img2, 'Image B', (220, 380), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0,0,0), 2)

# Chuyển sang grayscale
gray1 = cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY)
gray2 = cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY)

print('Đã tạo 2 ảnh demo thành công!')
print(f'Kích thước ảnh 1: {img1.shape}')
print(f'Kích thước ảnh 2: {img2.shape}')

In [ ]:
# Hiển thị 2 ảnh gốc
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].imshow(cv2.cvtColor(img1, cv2.COLOR_BGR2RGB))
axes[0].set_title('Ảnh 1 (Image A)', fontsize=13)
axes[0].axis('off')

axes[1].imshow(cv2.cvtColor(img2, cv2.COLOR_BGR2RGB))
axes[1].set_title('Ảnh 2 (Image B - xoay + nhiễu)', fontsize=13)
axes[1].axis('off')

plt.suptitle('2 ảnh đầu vào cho bài toán Feature Matching', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Khởi tạo ORB detector
orb = cv2.ORB_create(nfeatures=500)

# Trích xuất keypoints và descriptors
kp1, des1 = orb.detectAndCompute(gray1, None)
kp2, des2 = orb.detectAndCompute(gray2, None)

print(f'Số keypoints trong ảnh 1: {len(kp1)}')
print(f'Số keypoints trong ảnh 2: {len(kp2)}')
print(f'Kích thước descriptor: {des1.shape}')

In [ ]:
# Vẽ keypoints lên ảnh
img1_kp = cv2.drawKeypoints(img1.copy(), kp1, None, color=(0, 200, 0),
                             flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
img2_kp = cv2.drawKeypoints(img2.copy(), kp2, None, color=(0, 200, 0),
                             flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(cv2.cvtColor(img1_kp, cv2.COLOR_BGR2RGB))
axes[0].set_title(f'Keypoints Ảnh 1: {len(kp1)} điểm', fontsize=12)
axes[0].axis('off')

axes[1].imshow(cv2.cvtColor(img2_kp, cv2.COLOR_BGR2RGB))
axes[1].set_title(f'Keypoints Ảnh 2: {len(kp2)} điểm', fontsize=12)
axes[1].axis('off')

plt.suptitle('Phát hiện Keypoints bằng ORB', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Sử dụng Brute-Force Matcher với Hamming distance (phù hợp với ORB)
bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)

# Thực hiện matching
matches_raw = bf.match(des1, des2)

# Sắp xếp theo khoảng cách (nhỏ hơn = tốt hơn)
matches_raw = sorted(matches_raw, key=lambda x: x.distance)

print(f'Tổng số matches tìm được: {len(matches_raw)}')
print(f'Khoảng cách trung bình: {np.mean([m.distance for m in matches_raw]):.2f}')

In [ ]:
# Vẽ kết quả matching (lấy 50 matches đầu)
img_matches_raw = cv2.drawMatches(
    img1, kp1, img2, kp2,
    matches_raw[:50], None,
    flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS
)

plt.figure(figsize=(16, 6))
plt.imshow(cv2.cvtColor(img_matches_raw, cv2.COLOR_BGR2RGB))
plt.title(f'Brute-Force Matching (hiển thị 50/{len(matches_raw)} matches)', fontsize=13)
plt.axis('off')
plt.tight_layout()
plt.show()

**Nhận xét sau bước đối sánh cơ bản:**

- ORB tìm được khá nhiều keypoints trên cả 2 ảnh.
- Brute-Force Matcher ghép cặp các điểm đặc trưng dựa trên Hamming distance.
- Tuy nhiên, vẫn còn nhiều cặp ghép sai (outliers) do nhiễu ảnh và sự biến đổi phép chiếu.
- Cần áp dụng thêm các kỹ thuật lọc để cải thiện chất lượng đối sánh.

---
## Phần 3: Cải thiện kết quả đối sánh

Sử dụng 2 kỹ thuật:
1. **Ratio Test (Lowe's ratio test)**: lọc các cặp không rõ ràng
2. **RANSAC**: loại bỏ outliers, tìm homography tốt hơn

In [ ]:
# Dùng kNN Matcher để có thể áp dụng ratio test
bf_knn = cv2.BFMatcher(cv2.NORM_HAMMING)
matches_knn = bf_knn.knnMatch(des1, des2, k=2)

# Áp dụng Lowe's ratio test
# Nếu khoảng cách match tốt nhất < 0.75 * match tốt thứ 2 => giữ lại
good_matches = []
for m, n in matches_knn:
    if m.distance < 0.75 * n.distance:
        good_matches.append(m)

print(f'Matches trước ratio test : {len(matches_knn)}')
print(f'Matches sau ratio test   : {len(good_matches)}')
print(f'Tỉ lệ giữ lại: {len(good_matches)/len(matches_knn)*100:.1f}%')

In [ ]:
# Áp dụng RANSAC để loại bỏ outliers
# Cần ít nhất 4 điểm để tính homography

if len(good_matches) >= 4:
    # Lấy tọa độ các điểm
    src_pts = np.float32([kp1[m.queryIdx].pt for m in good_matches]).reshape(-1, 1, 2)
    dst_pts = np.float32([kp2[m.trainIdx].pt for m in good_matches]).reshape(-1, 1, 2)

    # Tìm homography bằng RANSAC
    H, mask = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, 5.0)

    # Lọc ra các inliers
    inlier_matches = [good_matches[i] for i in range(len(good_matches)) if mask[i][0] == 1]

    print(f'Matches sau ratio test : {len(good_matches)}')
    print(f'Matches sau RANSAC     : {len(inlier_matches)}')
    print(f'Số outliers bị loại   : {len(good_matches) - len(inlier_matches)}')
else:
    print('Không đủ điểm để tính RANSAC!')
    inlier_matches = good_matches

In [ ]:
# So sánh trước và sau khi lọc
img_before = cv2.drawMatches(
    img1, kp1, img2, kp2, matches_raw[:50], None,
    flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS
)

img_after = cv2.drawMatches(
    img1, kp1, img2, kp2, inlier_matches, None,
    matchColor=(0, 255, 0),
    flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS
)

fig, axes = plt.subplots(2, 1, figsize=(16, 10))

axes[0].imshow(cv2.cvtColor(img_before, cv2.COLOR_BGR2RGB))
axes[0].set_title(f'TRƯỚC khi lọc - BFMatcher (hiển thị 50 matches)', fontsize=12)
axes[0].axis('off')

axes[1].imshow(cv2.cvtColor(img_after, cv2.COLOR_BGR2RGB))
axes[1].set_title(f'SAU khi lọc - Ratio Test + RANSAC ({len(inlier_matches)} inlier matches)', fontsize=12)
axes[1].axis('off')

plt.suptitle('So sánh trước và sau khi cải thiện đối sánh', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**Nhận xét sau khi cải thiện:**

- **Ratio Test**: loại bỏ các cặp điểm mà sự khác biệt giữa match tốt nhất và thứ hai không rõ ràng, giảm thiểu các false positive.
- **RANSAC**: sử dụng mô hình hình học (homography) để loại bỏ những cặp điểm không phù hợp với phép biến đổi tổng thể.
- Kết quả sau khi lọc rõ ràng hơn nhiều, các đường kết nối chủ yếu là chính xác.
- Số lượng matches giảm nhưng chất lượng tăng lên đáng kể.

---
## Phần 4: Xây dựng mô hình CNN cho Photo Matching

Chúng ta xây dựng một mạng CNN đơn giản bằng PyTorch để trích xuất vector đặc trưng (feature vector) từ ảnh đầu vào.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class FeatureCNN(nn.Module):
    """
    CNN đơn giản để trích xuất đặc trưng từ ảnh.
    Input : ảnh RGB kích thước (3, 128, 128)
    Output: vector đặc trưng kích thước 128
    """
    def __init__(self, embedding_dim=128):
        super(FeatureCNN, self).__init__()

        # Khối Convolutional - trích xuất đặc trưng cục bộ
        self.conv_block = nn.Sequential(
            # Conv 1: 3 -> 32 channels
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),           # 128x128 -> 64x64

            # Conv 2: 32 -> 64 channels
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),           # 64x64 -> 32x32

            # Conv 3: 64 -> 128 channels
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),           # 32x32 -> 16x16

            # Conv 4: 128 -> 256 channels
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((4, 4))  # -> 4x4 bất kể input size
        )

        # Fully Connected - tạo embedding vector
        self.fc_block = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 4 * 4, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(512, embedding_dim)
        )

    def forward(self, x):
        # Trích xuất đặc trưng bằng conv
        features = self.conv_block(x)
        # Tạo embedding vector
        embedding = self.fc_block(features)
        # Chuẩn hóa L2 để dùng cosine similarity
        embedding = F.normalize(embedding, p=2, dim=1)
        return embedding


# Khởi tạo và kiểm tra
cnn = FeatureCNN(embedding_dim=128).to(device)
print('Kiến trúc FeatureCNN:')
print(cnn)

# Test với tensor giả
dummy_input = torch.randn(4, 3, 128, 128).to(device)  # batch=4, RGB, 128x128
output = cnn(dummy_input)
print(f'\nKích thước output: {output.shape}')  # Kỳ vọng: (4, 128)
print(f'Norm của vector đầu tiên: {output[0].norm().item():.4f}')  # Phải = 1 sau chuẩn hóa L2

In [ ]:
# Đếm số tham số của mô hình
total_params = sum(p.numel() for p in cnn.parameters())
trainable_params = sum(p.numel() for p in cnn.parameters() if p.requires_grad)

print(f'Tổng số tham số    : {total_params:,}')
print(f'Tham số có thể huấn luyện: {trainable_params:,}')

---
## Phần 5: Xây dựng Siamese Network

**Siamese Network** là kiến trúc gồm 2 nhánh CNN **dùng chung trọng số**. Mục tiêu là học một không gian embedding sao cho:
- Các cặp ảnh **giống nhau** có embedding **gần nhau**
- Các cặp ảnh **khác nhau** có embedding **xa nhau**

In [ ]:
class SiameseNetwork(nn.Module):
    """
    Siamese Network: 2 nhánh CNN dùng chung trọng số.
    Input : 2 ảnh (img1, img2)
    Output: khoảng cách Euclidean giữa 2 embeddings
    """
    def __init__(self, embedding_dim=128):
        super(SiameseNetwork, self).__init__()
        # Chỉ tạo 1 CNN, cả 2 nhánh sẽ dùng chung đối tượng này
        self.backbone = FeatureCNN(embedding_dim)

    def forward_one(self, x):
        """ Cho 1 ảnh qua CNN để lấy embedding """
        return self.backbone(x)

    def forward(self, img1, img2):
        """ Tính khoảng cách giữa 2 ảnh """
        emb1 = self.forward_one(img1)   # embedding ảnh 1
        emb2 = self.forward_one(img2)   # embedding ảnh 2

        # Tính Euclidean distance
        distance = F.pairwise_distance(emb1, emb2, p=2)
        return distance, emb1, emb2

    def get_similarity(self, img1, img2):
        """ Tính Cosine similarity giữa 2 ảnh (0 đến 1) """
        emb1 = self.forward_one(img1)
        emb2 = self.forward_one(img2)
        similarity = F.cosine_similarity(emb1, emb2)
        return similarity


# Khởi tạo Siamese Network
siamese = SiameseNetwork(embedding_dim=128).to(device)

# Test với 2 cấu hình ảnh
img_a = torch.randn(2, 3, 128, 128).to(device)  # batch=2
img_b = torch.randn(2, 3, 128, 128).to(device)

dist, emb_a, emb_b = siamese(img_a, img_b)
sim = siamese.get_similarity(img_a, img_b)

print(f'Output shape embedding: {emb_a.shape}')
print(f'Khoảng cách Euclidean : {dist.detach().cpu().numpy()}')
print(f'Độ tương đồng Cosine : {sim.detach().cpu().numpy()}')

---
## Phần 6: Dataset cho Siamese Network

Tạo dataset gồm 2 loại cặp:
- **Positive pair** (label = 0): 2 ảnh cùng class / cùng người
- **Negative pair** (label = 1): 2 ảnh khác class / khác người

In [ ]:
from torch.utils.data import Dataset
import random

class SiameseDataset(Dataset):
    """
    Dataset cho Siamese Network.
    Mỗi mẫu gồm (ảnh1, ảnh2, label):
      - label = 0: positive pair (giống nhau)
      - label = 1: negative pair (khác nhau)
    """
    def __init__(self, images, labels, transform=None, num_samples=1000):
        """
        images : list các ảnh (numpy array hoặc tensor)
        labels : list nhãn tương ứng
        """
        self.images = images
        self.labels = labels
        self.transform = transform
        self.num_samples = num_samples

        # Tạo index theo từng class
        self.class_indices = {}
        for idx, label in enumerate(labels):
            if label not in self.class_indices:
                self.class_indices[label] = []
            self.class_indices[label].append(idx)

        self.classes = list(self.class_indices.keys())

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        # Chọn ngẫu nhiên positive hay negative pair (50/50)
        is_positive = random.random() > 0.5

        if is_positive:
            # Positive pair: chọn 2 ảnh cùng class
            cls = random.choice(self.classes)
            idx1, idx2 = random.sample(self.class_indices[cls], 2) \
                if len(self.class_indices[cls]) >= 2 \
                else (self.class_indices[cls][0], self.class_indices[cls][0])
            pair_label = torch.tensor(0, dtype=torch.float32)  # 0 = giống
        else:
            # Negative pair: chọn 2 ảnh khác class
            cls1, cls2 = random.sample(self.classes, 2)
            idx1 = random.choice(self.class_indices[cls1])
            idx2 = random.choice(self.class_indices[cls2])
            pair_label = torch.tensor(1, dtype=torch.float32)  # 1 = khác

        img1 = self.images[idx1]
        img2 = self.images[idx2]

        if self.transform:
            img1 = self.transform(img1)
            img2 = self.transform(img2)

        return img1, img2, pair_label


print('Class SiameseDataset đã được định nghĩa!')

In [ ]:
from PIL import Image

# Tạo dữ liệu giả lập: 5 class, mỗi class 20 ảnh
# Mỗi class có màu nền khác nhau để phân biệt
def create_synthetic_images(num_classes=5, samples_per_class=20, size=128):
    images = []
    labels = []
    colors = [(200,80,80), (80,200,80), (80,80,200), (200,200,80), (200,80,200)]

    for cls in range(num_classes):
        for s in range(samples_per_class):
            # Tạo ảnh với màu nền theo class, thêm nhiễu nhỏ
            img = np.ones((size, size, 3), dtype=np.uint8)
            r, g, b = colors[cls]
            img[:, :, 0] = np.clip(r + np.random.randint(-30, 30), 0, 255)
            img[:, :, 1] = np.clip(g + np.random.randint(-30, 30), 0, 255)
            img[:, :, 2] = np.clip(b + np.random.randint(-30, 30), 0, 255)

            # Vẽ hình dạng theo class
            cx = size // 2 + np.random.randint(-10, 10)
            cy = size // 2 + np.random.randint(-10, 10)
            cv2.circle(img, (cx, cy), 20 + cls * 5, (255, 255, 255), -1)

            images.append(Image.fromarray(img))
            labels.append(cls)

    return images, labels


synthetic_imgs, synthetic_labels = create_synthetic_images()

# Transform: chuyển ảnh sang tensor
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

# Tạo dataset và dataloader
dataset = SiameseDataset(synthetic_imgs, synthetic_labels,
                         transform=transform, num_samples=500)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

print(f'Tổng số ảnh: {len(synthetic_imgs)}')
print(f'Số class   : {len(set(synthetic_labels))}')
print(f'Số sample  : {len(dataset)}')
print(f'Số batch   : {len(dataloader)}')

# Lấy 1 batch để kiểm tra
batch_img1, batch_img2, batch_label = next(iter(dataloader))
print(f'\nKích thước batch img1 : {batch_img1.shape}')
print(f'Kích thước batch label: {batch_label.shape}')
print(f'Tỉ lệ positive pairs  : {(batch_label==0).float().mean().item():.2f}')

In [ ]:
# Hiển thị một số cặp ảnh
fig, axes = plt.subplots(2, 6, figsize=(16, 6))

mean = np.array([0.5, 0.5, 0.5])
std = np.array([0.5, 0.5, 0.5])

for i in range(6):
    img1_show = batch_img1[i].permute(1, 2, 0).numpy()
    img2_show = batch_img2[i].permute(1, 2, 0).numpy()
    img1_show = np.clip(img1_show * std + mean, 0, 1)
    img2_show = np.clip(img2_show * std + mean, 0, 1)

    lbl = 'Positive (giống)' if batch_label[i].item() == 0 else 'Negative (khác)'
    color = 'green' if batch_label[i].item() == 0 else 'red'

    axes[0][i].imshow(img1_show)
    axes[0][i].set_title(f'Cặp {i+1}\n{lbl}', fontsize=8, color=color)
    axes[0][i].axis('off')

    axes[1][i].imshow(img2_show)
    axes[1][i].axis('off')

    axes[0][0].set_ylabel('Ảnh 1', fontsize=11)
    axes[1][0].set_ylabel('Ảnh 2', fontsize=11)

plt.suptitle('Ví dụ các cặp ảnh trong SiameseDataset', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Phần 7: Loss Function và Huấn luyện

Sử dụng **Contrastive Loss** - loss function phù hợp cho Siamese Network:

$$L = (1 - y) \cdot \frac{1}{2} D^2 + y \cdot \frac{1}{2} \max(0, m - D)^2$$

Trong đó:
- $D$: khoảng cách Euclidean giữa 2 embeddings
- $y = 0$: positive pair (giống nhau) - cần $D$ nhỏ
- $y = 1$: negative pair (khác nhau) - cần $D > m$ (margin)

In [ ]:
class ContrastiveLoss(nn.Module):
    """
    Contrastive Loss cho Siamese Network.
    margin: ngưỡng khoảng cách tối thiểu cho negative pairs
    """
    def __init__(self, margin=1.0):
        super(ContrastiveLoss, self).__init__()
        self.margin = margin

    def forward(self, distance, label):
        """
        distance: khoảng cách Euclidean (batch)
        label   : 0 = giống, 1 = khác
        """
        # Positive pair: cần khoảng cách nhỏ
        loss_pos = (1 - label) * 0.5 * distance.pow(2)

        # Negative pair: cần khoảng cách > margin
        loss_neg = label * 0.5 * torch.clamp(self.margin - distance, min=0).pow(2)

        loss = (loss_pos + loss_neg).mean()
        return loss


print('Contrastive Loss đã định nghĩa xong!')

# Kiểm tra nhanh
criterion_test = ContrastiveLoss(margin=1.0)
d = torch.tensor([0.2, 0.8, 1.5, 0.1])
y = torch.tensor([0.0, 1.0, 0.0, 1.0])  # 0=giống, 1=khác
loss_val = criterion_test(d, y)
print(f'Loss test: {loss_val.item():.4f}')

In [ ]:
# Khởi tạo model, loss, optimizer
siamese_model = SiameseNetwork(embedding_dim=128).to(device)
criterion = ContrastiveLoss(margin=1.0)
optimizer = optim.Adam(siamese_model.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

NUM_EPOCHS = 15
loss_history = []

print('Bắt đầu huấn luyện...')
print('='*55)

for epoch in range(NUM_EPOCHS):
    siamese_model.train()
    epoch_loss = 0.0
    num_batches = 0

    for img1_batch, img2_batch, label_batch in dataloader:
        # Chuyển dữ liệu lên device
        img1_batch = img1_batch.to(device)
        img2_batch = img2_batch.to(device)
        label_batch = label_batch.to(device)

        # Forward pass
        optimizer.zero_grad()
        distance, _, _ = siamese_model(img1_batch, img2_batch)

        # Tính loss
        loss = criterion(distance, label_batch)

        # Backward + update
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        num_batches += 1

    # Tính trung bình loss trong epoch
    avg_loss = epoch_loss / num_batches
    loss_history.append(avg_loss)
    scheduler.step()

    if (epoch + 1) % 3 == 0 or epoch == 0:
        lr_now = optimizer.param_groups[0]['lr']
        print(f'Epoch [{epoch+1:>2}/{NUM_EPOCHS}] | Loss: {avg_loss:.4f} | LR: {lr_now:.6f}')

print('='*55)
print('Huấn luyện hoàn thành!')

In [ ]:
# Vẽ biểu đồ loss theo epoch
plt.figure(figsize=(10, 5))
plt.plot(range(1, NUM_EPOCHS + 1), loss_history, 'b-o', linewidth=2, markersize=6, label='Training Loss')
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Contrastive Loss', fontsize=12)
plt.title('Training Loss của Siamese Network', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.xticks(range(1, NUM_EPOCHS + 1))
plt.tight_layout()
plt.show()

print(f'Loss ban đầu : {loss_history[0]:.4f}')
print(f'Loss cuối    : {loss_history[-1]:.4f}')
print(f'Mức giảm     : {(loss_history[0] - loss_history[-1]) / loss_history[0] * 100:.1f}%')

---
## Phần 8: Đánh giá mô hình

Kiểm tra mô hình với một số cặp ảnh, in ra điểm tương đồng và dự đoán.

In [ ]:
def evaluate_pair(model, img1_pil, img2_pil, transform, threshold=0.5):
    """
    Đánh giá 1 cặp ảnh.
    Returns: similarity (0-1), dự đoán (Matched / Unknown)
    """
    model.eval()
    with torch.no_grad():
        t1 = transform(img1_pil).unsqueeze(0).to(device)
        t2 = transform(img2_pil).unsqueeze(0).to(device)

        # Tính cosine similarity
        similarity = siamese_model.get_similarity(t1, t2).item()

    prediction = 'Matched' if similarity > threshold else 'Unknown'
    return similarity, prediction


# Lấy một số ảnh mẫu để test
test_cases = [
    (0, 1, 'Positive (cùng class 0)'),
    (0, 5, 'Positive (cùng class 0)'),
    (0, 20, 'Negative (class 0 vs class 1)'),
    (0, 40, 'Negative (class 0 vs class 2)'),
    (20, 21, 'Positive (cùng class 1)'),
    (20, 60, 'Negative (class 1 vs class 3)'),
]

print(f'{"Cặp":<5} {"Mô tả":<35} {"Độ tương đồng":<15} {"Dự đoán"}')
print('-' * 65)

for i, (idx1, idx2, desc) in enumerate(test_cases):
    sim, pred = evaluate_pair(
        siamese_model,
        synthetic_imgs[idx1],
        synthetic_imgs[idx2],
        transform,
        threshold=0.5
    )
    print(f'{i+1:<5} {desc:<35} {sim:<15.4f} {pred}')

In [ ]:
# Hiển thị ảnh kết quả
fig, axes = plt.subplots(len(test_cases), 3, figsize=(12, 4 * len(test_cases)))

mean_arr = np.array([0.5, 0.5, 0.5])
std_arr = np.array([0.5, 0.5, 0.5])

for row, (idx1, idx2, desc) in enumerate(test_cases):
    sim, pred = evaluate_pair(
        siamese_model, synthetic_imgs[idx1], synthetic_imgs[idx2], transform
    )

    # Ảnh 1
    t1 = transform(synthetic_imgs[idx1]).permute(1, 2, 0).numpy()
    t1 = np.clip(t1 * std_arr + mean_arr, 0, 1)

    # Ảnh 2
    t2 = transform(synthetic_imgs[idx2]).permute(1, 2, 0).numpy()
    t2 = np.clip(t2 * std_arr + mean_arr, 0, 1)

    axes[row][0].imshow(t1); axes[row][0].set_title('Ảnh 1'); axes[row][0].axis('off')
    axes[row][1].imshow(t2); axes[row][1].set_title('Ảnh 2'); axes[row][1].axis('off')

    color = 'green' if pred == 'Matched' else 'red'
    axes[row][2].text(0.5, 0.5, f'{desc}\n\nĐộ tương đồng: {sim:.4f}\nDự đoán: {pred}',
                      transform=axes[row][2].transAxes,
                      ha='center', va='center', fontsize=11, color=color,
                      bbox=dict(boxstyle='round,pad=0.5', facecolor='lightyellow', alpha=0.8))
    axes[row][2].axis('off')

plt.suptitle('Kết quả đánh giá mô hình Siamese Network', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Phần 9 & 10: Nhận diện khuôn mặt thời gian thực

Hệ thống nhận diện khuôn mặt thời gian thực sử dụng:
- **MTCNN**: phát hiện khuôn mặt
- **FaceNet (InceptionResnetV1)**: trích xuất embedding
- **Cosine Similarity**: so sánh với ảnh mẫu

In [ ]:
# Kiểm tra facenet-pytorch
try:
    from facenet_pytorch import MTCNN, InceptionResnetV1
    print('facenet-pytorch đã sẵn sàng!')
    FACENET_AVAILABLE = True
except ImportError:
    print('Chưa cài facenet-pytorch. Chạy: pip install facenet-pytorch')
    FACENET_AVAILABLE = False

In [ ]:
class FaceRecognitionSystem:
    """
    Hệ thống nhận diện khuôn mặt thời gian thực.
    - Detect: MTCNN
    - Embed : FaceNet
    - Match : Cosine Similarity
    """
    def __init__(self, threshold=0.7, device='cpu'):
        self.threshold = threshold
        self.device = device
        self.reference_embedding = None
        self.reference_name = 'Unknown'

        if FACENET_AVAILABLE:
            # MTCNN: phát hiện và căn chỉnh khuôn mặt
            self.mtcnn = MTCNN(
                image_size=160,
                margin=20,
                keep_all=False,
                device=device
            )
            # FaceNet: trích xuất đặc trưng
            self.facenet = InceptionResnetV1(pretrained='vggface2').eval().to(device)
            print('Hệ thống nhận diện khuôn mặt khởi tạo thành công!')
        else:
            print('Không thể khởi tạo: facenet-pytorch chưa được cài đặt.')

    def extract_embedding(self, pil_image):
        """ Trích xuất embedding từ ảnh PIL """
        face_tensor = self.mtcnn(pil_image)
        if face_tensor is None:
            return None
        face_tensor = face_tensor.unsqueeze(0).to(self.device)
        with torch.no_grad():
            embedding = self.facenet(face_tensor)
        return F.normalize(embedding, p=2, dim=1)

    def register_reference(self, pil_image, name='Person'):
        """ Đăng ký ảnh mẫu để so sánh """
        emb = self.extract_embedding(pil_image)
        if emb is not None:
            self.reference_embedding = emb
            self.reference_name = name
            print(f'Đã đăng ký ảnh mẫu cho: {name}')
            return True
        print('Không phát hiện khuôn mặt trong ảnh mẫu!')
        return False

    def identify(self, pil_image):
        """
        Nhận diện khuôn mặt trong ảnh.
        Returns: (boxes, label, similarity)
        """
        boxes, _ = self.mtcnn.detect(pil_image)
        if boxes is None:
            return None, 'No Face', 0.0

        current_emb = self.extract_embedding(pil_image)
        if current_emb is None or self.reference_embedding is None:
            return boxes, 'Unknown', 0.0

        # Tính cosine similarity
        similarity = F.cosine_similarity(
            self.reference_embedding, current_emb
        ).item()

        # So sánh với ngưỡng
        label = self.reference_name if similarity > self.threshold else 'Unknown'
        return boxes, label, similarity


print('Class FaceRecognitionSystem đã định nghĩa!')

In [ ]:
def draw_face_result(frame, boxes, label, similarity):
    """
    Vẽ bounding box và label lên frame.
    Màu xanh = Matched, đỏ = Unknown.
    """
    if boxes is None:
        return frame

    result = frame.copy()
    for box in boxes:
        x1, y1, x2, y2 = [int(v) for v in box]
        color = (0, 200, 0) if 'Unknown' not in label else (0, 0, 200)

        # Vẽ bounding box
        cv2.rectangle(result, (x1, y1), (x2, y2), color, 2)

        # Hiển thị label và similarity
        text = f'{label} ({similarity:.2f})'
        cv2.rectangle(result, (x1, y1-30), (x1+len(text)*10, y1), color, -1)
        cv2.putText(result, text, (x1, y1-8),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

    return result


def run_realtime_recognition(system, source=0, duration_seconds=30):
    """
    Chạy nhận diện khuôn mặt thời gian thực từ webcam.
    Nhấn 'q' để thoát.
    source: 0 = webcam mặc định
    """
    cap = cv2.VideoCapture(source)
    if not cap.isOpened():
        print('Không thể mở webcam!')
        return

    print('Bắt đầu nhận diện thời gian thực... Nhấn Q để thoát.')
    frame_count = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            print('Không đọc được frame!')
            break

        # Chỉ xử lý mỗi 3 frame để tăng FPS
        if frame_count % 3 == 0:
            pil_frame = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            boxes, label, sim = system.identify(pil_frame)
            frame = draw_face_result(frame, boxes, label, sim)

        # Hiển thị FPS
        cv2.putText(frame, f'Frame: {frame_count}', (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)

        cv2.imshow('Face Recognition - Nhấn Q để thoát', frame)
        frame_count += 1

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()
    print('Đã kết thúc thời gian thực!')


print('Các hàm hỗ trợ đã định nghĩa!')

---
## Phần 10: Demo

Để chạy demo thời gian thực, thực hiện các bước sau:

**Bước 1**: Khởi tạo hệ thống  
**Bước 2**: Đăng ký ảnh mẫu (chụp ảnh từ webcam hoặc dùng ảnh có sẵn)  
**Bước 3**: Chạy nhận diện thời gian thực

In [ ]:
# Khởi tạo hệ thống nhận diện
if FACENET_AVAILABLE:
    face_system = FaceRecognitionSystem(threshold=0.7, device=str(device))
else:
    print('Facenet không khả dụng, bỏ qua phần này.')
    face_system = None

In [ ]:
# Đăng ký ảnh mẫu từ file ảnh (thay đường dẫn nếu cần)
# Ví dụ: face_system.register_reference(Image.open('reference.jpg'), name='Nguyen Van A')

# Demo mang tính minh họa (dùng ảnh giả)
if face_system is not None:
    demo_ref = Image.fromarray(
        np.random.randint(100, 200, (200, 200, 3), dtype=np.uint8)
    )
    print('Hướng dẫn sử dụng:')
    print('1. Đăng ký ảnh mẫu: face_system.register_reference(Image.open("ten_anh.jpg"), name="Ten")')
    print('2. Chạy webcam    : run_realtime_recognition(face_system, source=0)')

In [ ]:
# Chạy thời gian thực (bỏ comment để bắt đầu)
# Cần có webcam và ảnh mẫu đã được đăng ký

# if face_system is not None:
#     run_realtime_recognition(face_system, source=0)

print('Bỏ comment 2 dòng trên để bắt đầu webcam thời gian thực!')

In [ ]:
# Minh họa pipeline nhận diện (không cần webcam)
# Tạo ảnh giả lập để demo flow

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Ảnh 1: tạo ảnh khuôn mặt giả
demo_face = np.ones((300, 250, 3), dtype=np.uint8) * 220
# Vẽ khuôn mặt
cv2.ellipse(demo_face, (125, 130), (80, 100), 0, 0, 360, (210, 170, 130), -1)
cv2.ellipse(demo_face, (90, 110), (15, 12), 0, 0, 360, (60, 40, 30), -1)   # mat trai
cv2.ellipse(demo_face, (160, 110), (15, 12), 0, 0, 360, (60, 40, 30), -1)  # mat phai
cv2.ellipse(demo_face, (125, 165), (30, 12), 0, 0, 180, (160, 80, 80), 2)  # mieng
cv2.rectangle(demo_face, (50, 20), (200, 250), (0, 200, 0), 2)  # bbox
cv2.putText(demo_face, 'Matched (0.85)', (45, 15),
            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 200, 0), 1)

axes[0].imshow(cv2.cvtColor(demo_face, cv2.COLOR_BGR2RGB))
axes[0].set_title('Bước 1: Phát hiện khuôn mặt\n(MTCNN)', fontsize=11)
axes[0].axis('off')

# Ảnh 2: embedding vector
axes[1].bar(range(20), np.random.randn(20), color='steelblue', alpha=0.7)
axes[1].set_title('Bước 2: Trích xuất embedding\n(FaceNet - 512 dims)', fontsize=11)
axes[1].set_xlabel('Chiều embedding (minh họa)')
axes[1].set_ylabel('Giá trị')

# Ảnh 3: so sánh
ref_emb = np.random.randn(20)
cur_emb = ref_emb + np.random.randn(20) * 0.2  # gan giong
axes[2].plot(ref_emb, 'r-o', label='Embedding mẫu', markersize=4)
axes[2].plot(cur_emb, 'b-o', label='Embedding hiện tại', markersize=4)
axes[2].set_title('Bước 3: So sánh Cosine Similarity\n-> Dự đoán: Khớp', fontsize=11)
axes[2].legend()
axes[2].set_xlabel('Chiều embedding')

plt.suptitle('Pipeline nhận diện khuôn mặt thời gian thực', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Phần 11: Nhận xét cuối bài

### So sánh Phương pháp Truyền thống vs Deep Learning

| Tiêu chí | Truyền thống (ORB/SIFT) | Deep Learning (CNN/FaceNet) |
|---|---|---|
| Tốc độ xử lý | Nhanh, nhẹ | Chậm hơn, cần GPU |
| Độ chính xác | Vừa đủ | Cao hơn nhiều |
| Dữ liệu cần | Không cần huấn luyện | Cần tập dữ liệu |
| Khả năng tổng quát | Thấp, phụ thuộc texture | Cao, học đặc trưng trừu tượng |
| Xử lý ảnh biến đổi mạnh | Hạn chế | Tốt hơn |
| Giải thích được | Dễ giải thích | Khó giải thích (blackbox) |
| Triển khai | Đơn giản, không cần GPU | Phức tạp hơn |

### Ưu nhược điểm

**Phương pháp truyền thống:**
- Ưu: nhanh, không cần dữ liệu huấn luyện, dễ hiểu và triển khai
- Nhược: kém chính xác khi ảnh có nhiều biến đổi lớn (sáng, góc nhìn, biến dạng)

**Deep Learning (Siamese/FaceNet):**
- Ưu: chính xác cao, tổng quát tốt, hiệu quả với dữ liệu phức tạp
- Nhược: cần nhiều dữ liệu huấn luyện, chi phí tính toán lớn, khó giải thích

### Khi nào nên dùng mỗi phương pháp?

**Dùng phương pháp truyền thống khi:**
- Không có đủ dữ liệu để huấn luyện deep learning
- Ứng dụng cần tốc độ cao, tài nguyên hạn chế (edge device)
- Ảnh có cấu trúc rõ ràng (tài liệu, bản đồ, logo)
- Muốn kết quả dễ giải thích, debug

**Dùng Deep Learning khi:**
- Có đủ dữ liệu huấn luyện
- Độ chính xác là ưu tiên hàng đầu
- Ảnh phức tạp, nhiều điều kiện ánh sáng, góc độ khác nhau
- Bài toán nhận diện người, biển số xe, sản phẩm...

In [ ]:
# Biểu đồ so sánh tổng quan
categories = ['Độ chính xác', 'Tốc độ', 'Dễ triển khai', 'Tài nguyên', 'Khả năng\ntổng quát']
traditional = [5, 9, 9, 8, 4]
deep_learning = [9, 6, 5, 4, 9]

x = np.arange(len(categories))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
bars1 = ax.bar(x - width/2, traditional, width, label='Phương pháp truyền thống', color='steelblue', alpha=0.8)
bars2 = ax.bar(x + width/2, deep_learning, width, label='Deep Learning', color='tomato', alpha=0.8)

ax.set_xlabel('Tiêu chí đánh giá', fontsize=12)
ax.set_ylabel('Điểm (1-10)', fontsize=12)
ax.set_title('So sánh Phương pháp Truyền thống vs Deep Learning', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(categories, fontsize=11)
ax.legend(fontsize=11)
ax.set_ylim(0, 11)
ax.grid(axis='y', alpha=0.3)

# Vẽ giá trị trên cột
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.1,
            str(int(bar.get_height())), ha='center', va='bottom', fontweight='bold')
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.1,
            str(int(bar.get_height())), ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

---

## Kết luận

Qua bài thực hành này, chúng ta đã hoàn thành:

1. Feature matching bằng ORB + BFMatcher, cải thiện bằng ratio test và RANSAC
2. Xây dựng CNN trích xuất embedding vector
3. Xây dựng Siamese Network học sự tương đồng giữa 2 ảnh
4. Huấn luyện với Contrastive Loss
5. Xây dựng pipeline nhận diện khuôn mặt với MTCNN + FaceNet

Bài thực hành giúp chúng ta hiểu sâu hơn về cả hai hướng tiếp cận: truyền thống dựa trên feature descriptor và hiện đại dựa trên deep learning. Mỗi phương pháp có ưu nhược điểm riêng và phù hợp với các tình huống sử dụng khác nhau.